# 🎮 Crafter x DreamerV3 训练与评测指南 (本地 & Google Colab 双环境自适应)

本 Notebook 专为在 **本地计算机 (Windows / WSL / Linux)** 或 **Google Colab (免费 T4 / V100 / A100 GPU)** 上训练 **DreamerV3** 强化学习世界模型而设计：
- 🌟 **双环境自适应支持**：智能检测运行环境，在本地 IDE (VS Code / Antigravity) 与 Colab 网页端均可开箱即跑！
- 🔄 **一键断点续训 (Resume Training)**：
  - **Colab 模式**：自动挂载 Google Drive，检测 `crafter_dreamerv3_100k` 历史检查点并自动恢复至本地 SSD 继续训练。
  - **本地运行模式**：自动检测本地 `output/crafter_dreamerv3_100k`、下载压缩包或云端同步盘，无缝接续已有进度 (如 65,190 步)！
- 📊 **训练全景看板与成果评测**：动态绘制得分与存活曲线，展示 Crafter 22 项成就解锁率柱状图。


## 1. 检查计算设备与 GPU 加速环境
- **Google Colab 用户**：Colab 菜单 -> **代码执行程序 (Runtime)** -> **更改运行时类型 (Change runtime type)** -> 硬件加速器选择 **T4 GPU**。
- **本地用户**：自动检测本地可用硬件与计算设备。


In [1]:
import shutil
import subprocess

# 1. 检查并打印 GPU 硬件状态 (显式捕获输出以兼容本地与云端环境)
if shutil.which("nvidia-smi"):
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout:
        print(res.stdout, flush=True)
    if res.stderr:
        print(res.stderr, flush=True)
else:
    print("未检测到 nvidia-smi 工具（当前实例未分配 GPU 硬件或处于纯 CPU 运行时）。", flush=True)

# 2. 检查 JAX 加速设备
print("\n--- JAX 计算设备检测 ---", flush=True)
try:
    import jax
    devices = jax.devices()
    print("当前 JAX 可用计算设备:", devices, flush=True)
    has_gpu = any(d.platform == "gpu" for d in devices)
except ImportError:
    has_gpu = False
    print("当前环境尚未安装 JAX，将在后续依赖安装步骤中自动配置。", flush=True)

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if not has_gpu:
    print("\n" + "!" * 70, flush=True)
    if in_colab:
        print("【提示】当前运行在 Google Colab 但处于 CPU 模式！", flush=True)
        print("👉 顶部菜单: 代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)", flush=True)
        print("👉 硬件加速器选择: T4 GPU -> 点击 保存 (Save)", flush=True)
    else:
        print("【提示】当前运行在本地计算机环境（处于 CPU 设备模式）：", flush=True)
        print("• 官方 JAX 在 Windows 原生系统仅提供 CPU 版本，适合参数验证、小步数调试或离线指标可视化。", flush=True)
        print("• 若需要在本地使用 NVIDIA 显卡满速训练，推荐在 WSL 2 (Ubuntu) 环境中运行。", flush=True)
        print("• 若想利用免费云端 GPU 算力，可随时将本 Notebook 导入至 Google Colab 执行！", flush=True)
    print("!" * 70 + "\n", flush=True)
else:
    print("\n✅ GPU 加速器已就绪，可顺畅继续执行后续训练步骤！", flush=True)


未检测到 nvidia-smi 工具（当前实例未分配 GPU 硬件或处于纯 CPU 运行时）。

--- JAX 计算设备检测 ---
当前 JAX 可用计算设备: [CpuDevice(id=0)]

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
【提示】当前运行在 Google Colab 但处于 CPU 模式！
👉 顶部菜单: 代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)
👉 硬件加速器选择: T4 GPU -> 点击 保存 (Save)
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



## 2. 检查与恢复历史检查点 (支持本地/云端断点续训)
> 💡 **断点续训核心机制**：
> - **Google Colab 模式**：自动挂载 Google Drive，并检测 `我的云端硬盘/crafter_dreamerv3_100k` 是否存在历史存档。
> - **本地运行模式**：自动检测本地 `output/crafter_dreamerv3_100k`、zip 压缩包或 Google Drive 电脑版同步盘。
> - 若检测到已有权重或 `metrics.jsonl`，自动读取已完成步数（如 65,190 steps），自动准备从该步数向后继续训练！


In [2]:
import os
import sys
import json
import shutil
import zipfile
from pathlib import Path

# 1. 智能环境识别与路径配置
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

current_step = 0
resume_ready = False

if IN_COLAB:
    print("🌐 [运行环境] 检测到 Google Colab 云端实例")
    from google.colab import drive
    print("正在请求挂载 Google Drive...")
    drive.mount("/content/drive")
    
    gdrive_path = Path("/content/drive/MyDrive/crafter_dreamerv3_100k")
    local_path = Path("/content/output/crafter_dreamerv3_100k")
    local_path.mkdir(parents=True, exist_ok=True)
    
    if gdrive_path.exists():
        print(f"\n[检测到云盘目录] {gdrive_path}")
        ckpt_files = list(gdrive_path.glob("*.ckpt")) or list(gdrive_path.glob("ckpt/*"))
        has_metrics = (gdrive_path / "metrics.jsonl").exists()
        
        if ckpt_files or has_metrics:
            print("[恢复检查点] 正在从 Google Drive 恢复历史训练权重与经验回放数据至高速 SSD...")
            shutil.copytree(gdrive_path, local_path, dirs_exist_ok=True)
            print("[恢复完成] 历史数据已完整同步至运行环境。")
else:
    print("💻 [运行环境] 检测到本地计算机环境 (Windows / Linux)")
    # 定位本地项目根目录
    cur = Path.cwd()
    if (cur / "colab").exists():
        project_root = cur
    elif (cur.parent / "colab").exists():
        project_root = cur.parent
    else:
        project_root = cur
    
    local_path = project_root / "output" / "crafter_dreamerv3_100k"
    local_path.mkdir(parents=True, exist_ok=True)
    print(f"📁 本地训练输出目录: {local_path}")
    
    # 智能检查 1: 本地是否存在已下载的 zip 存档 (如 output/crafter_dreamerv3_100k.zip)
    zip_candidate = project_root / "output" / "crafter_dreamerv3_100k.zip"
    if zip_candidate.exists() and not (local_path / "metrics.jsonl").exists():
        print(f"📦 [自动解压] 发现云端下载压缩包: {zip_candidate}，正在解压至本地目录...")
        with zipfile.ZipFile(zip_candidate, "r") as zf:
            zf.extractall(local_path)
        print("✅ 解压完成！")
    
    # 智能检查 2: 检测 Google Drive 电脑版本地同步路径
    possible_drives = [
        Path("G:/我的云端硬盘/crafter_dreamerv3_100k"),
        Path("G:/My Drive/crafter_dreamerv3_100k"),
        Path.home() / "Google Drive" / "crafter_dreamerv3_100k",
    ]
    for p in possible_drives:
        if p.exists() and not (local_path / "metrics.jsonl").exists():
            print(f"🔄 [检测到云盘同步盘] 正在从 {p} 同步至本地工作目录...")
            shutil.copytree(p, local_path, dirs_exist_ok=True)
            break

# 2. 解析当前检查点与已完成步数
metrics_file = local_path / "metrics.jsonl"
ckpt_files = list(local_path.glob("*.ckpt")) or list(local_path.glob("ckpt/*"))

if metrics_file.exists() or ckpt_files:
    if metrics_file.exists():
        records = []
        with open(metrics_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        records.append(json.loads(line))
                    except Exception:
                        pass
        if records:
            current_step = int(records[-1].get("step", 0))
            print(f"📊 当前已完成历史训练步数: {current_step:,} steps")
            resume_ready = True
    else:
        print("📊 发现模型权重文件，已就绪恢复！")
        resume_ready = True
else:
    print(f"\n[全新训练] 当前目录暂无历史存档，本次将从 0 步开始全新训练。")
    if not IN_COLAB:
        print("\n" + "-" * 60)
        print("💡 【如何将云端的历史存档拉到本地继续训练】：")
        print(" 1. 登录 Google 云端硬盘 (Google Drive) 网页端：https://drive.google.com/drive/")
        print(" 2. 找到训练文件夹 'crafter_dreamerv3_100k'，右键 -> 点击【下载】")
        print(f" 3. 将下载的 zip 压缩包放入: {project_root / 'output' / 'crafter_dreamerv3_100k.zip'}")
        print(" 4. 重新运行本单元格，代码将自动解压并直接开启断点续训！")
        print("-" * 60)

print("\n" + "=" * 60)
if resume_ready:
    print(f"🚀 已就绪【断点续训】模式！历史进度: {current_step:,} steps")
    print(f"👉 下方训练单元格将自动从第 {current_step:,} 步向后继续训练！")
    print(f"👉 推荐将目标步数设置为 100,000 或 200,000 步")
else:
    print("🌱 已就绪【全新训练】模式！将从 0 步开始训练至目标步数 (默认 100,000 步)")
print("=" * 60)


Mounted at /content/drive

[检测到云盘目录] /content/drive/MyDrive/crafter_dreamerv3_100k
[恢复检查点] 正在从 Google Drive 恢复历史训练权重与经验回放数据至高速 SSD...


KeyboardInterrupt: 

## 3. 拉取 DreamerV3 官方源码与安装依赖
本步骤自动克隆官方 `danijar/dreamerv3` 仓库，安装 `crafter` 与核心依赖，并自动应用新版 JAX 关键字参数兼容补丁。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# 1. 定位与克隆官方 DreamerV3 仓库
try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    repo_dir = Path("/content/dreamerv3")
else:
    cur = Path.cwd()
    project_root = cur if (cur / "colab").exists() else cur.parent
    repo_dir = project_root / "colab" / "dreamerv3"

if not (repo_dir / "dreamerv3" / "main.py").exists():
    print(f"正在克隆官方 DreamerV3 仓库至: {repo_dir}...")
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/danijar/dreamerv3.git", str(repo_dir)], check=True)
    print("DreamerV3 克隆完成。")
else:
    print(f"DreamerV3 仓库已就绪: {repo_dir}")

# 2. 检查与补齐核心依赖包
pkgs = [
    ("crafter", "crafter"),
    ("elements", "elements>=3.19.1"),
    ("portal", "portal>=3.5.0"),
    ("ninjax", "ninjax>=3.5.1"),
    ("optax", "optax"),
    ("chex", "chex"),
    ("einops", "einops"),
    ("ruamel.yaml", "ruamel.yaml"),
    ("scope", "scope>=0.4.4"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
]
missing_pkgs = []
for import_name, install_name in pkgs:
    try:
        __import__(import_name.split(".")[0])
    except ImportError:
        missing_pkgs.append(install_name)

if missing_pkgs:
    print(f"正在安装缺失依赖: {' '.join(missing_pkgs)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)
else:
    print("所有核心依赖均已就绪。")

# 3. 自动注入新版 JAX jit 兼容补丁
main_file = repo_dir / "dreamerv3" / "main.py"
if main_file.exists():
    content = main_file.read_text(encoding="utf-8")
    patch_marker = "# === JAX COMPATIBILITY MONKEY PATCH ==="
    if patch_marker not in content:
        patch_lines = [
            patch_marker,
            "import jax",
            "_orig_jit = jax.jit",
            "def _compat_jit(fun=None, *args, **kwargs):",
            "    if not args:",
            "        return _orig_jit(fun, **kwargs)",
            "    names = ['in_shardings', 'out_shardings', 'static_argnums', 'static_argnames', 'donate_argnums']",
            "    for val, name in zip(args, names):",
            "        if val is not None and name not in kwargs and name != 'static_argnames':",
            "            kwargs[name] = val",
            "    return _orig_jit(fun, **kwargs)",
            "jax.jit = _compat_jit",
            "",
        ]
        patch_code = "\n".join(patch_lines)
        main_file.write_text(patch_code + "\n" + content, encoding="utf-8")
        print("已成功注入 JAX jit 兼容补丁！")

# 4. 自动移除新版 XLA 中已废弃的编译参数 (防止 Unknown flags in XLA_FLAGS)
internal_file = repo_dir / "embodied" / "jax" / "internal.py"
if internal_file.exists():
    i_content = internal_file.read_text(encoding="utf-8")
    replacements = [
        ("'--xla_gpu_enable_triton_softmax_fusion=false',", "# '--xla_gpu_enable_triton_softmax_fusion=false',"),
        ("'--xla_gpu_graph_level=0',", "# '--xla_gpu_graph_level=0',"),
    ]
    modified = False
    for old, new in replacements:
        if old in i_content:
            i_content = i_content.replace(old, new)
            modified = True
    if modified:
        internal_file.write_text(i_content, encoding="utf-8")
        print("已成功清除 JAX/XLA 废弃参数补丁！")

print("\n环境配置与源码补丁准备完毕！")


DreamerV3 仓库已就绪: /content/dreamerv3
所有核心依赖均已就绪。
已成功清除 JAX/XLA 废弃参数补丁！

环境配置与源码补丁准备完毕！


## 4. JAX 显存策略与环境变量配置
JAX 默认会直接预分配 90% 显存。在单 GPU 且需多任务协作的 Colab 上，关闭预分配以按需申请显存，提升稳定性。

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
# 清理系统环境中可能残留的旧版废弃 XLA 参数
if "XLA_FLAGS" in os.environ:
    del os.environ["XLA_FLAGS"]
print("JAX 显存优化参数已注入系统环境。")


JAX 显存优化参数已注入系统环境。


## 5. 启动 / 继续 DreamerV3 训练

### 📌 关键配置与继续训练说明:
- **`TARGET_STEPS` (目标总步数)**:
  - 若这是**首次训练**：推荐设为 `100000` (100k 步)。
  - 若这是**继续训练**：必须大于当前已完成步数！例如已完成 100k 步，继续训练推荐设为 `200000` (再练 100k 步，总计 200k 步)。
  - **DreamerV3 断点续训原理**：当 `--logdir` 指向包含已有检查点的目录时，DreamerV3 会自动无缝加载模型权重、优化器状态和经验池，直接从上次记录的 step 继续向后训练，直至达到 `--run.steps`。
- **自动云盘持久化安全机制**：
  - 训练结束或用户中断时，自动将最新权重、经验池与日志增量同步回 Google Drive (`/content/drive/MyDrive/crafter_dreamerv3_100k`)，防止掉线丢失！


In [ ]:
import os
import sys
import json
import time
import shutil
import threading
import subprocess
from pathlib import Path

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    output_dir = Path("/content/output/crafter_dreamerv3_100k")
    gdrive_dir = Path("/content/drive/MyDrive/crafter_dreamerv3_100k")
    repo_dir = Path("/content/dreamerv3")
else:
    cur = Path.cwd()
    project_root = cur if (cur / "colab").exists() else cur.parent
    output_dir = project_root / "output" / "crafter_dreamerv3_100k"
    gdrive_dir = Path("G:/我的云端硬盘/crafter_dreamerv3_100k")
    repo_dir = project_root / "colab" / "dreamerv3"

output_dir.mkdir(parents=True, exist_ok=True)

# ==================== 用户参数配置 ====================
# 目标总步数 (Total Environment Steps):
# 推荐设置为 100,000 或 200,000 步 (若已有 65k 步进度，设为 100,000 将增量跑 35k 步)
TARGET_STEPS = 100000

# 模型规模预设 (保持与历史一致，默认 size12m，显存较小时可选 size1m)
MODEL_SIZE = "size12m"

# 样本复用比率 (默认: 512)
TRAIN_RATIO = 512

# 途中自动备份到 Google Drive 的间隔时间 (秒，仅在云盘可用时生效)
AUTO_BACKUP_INTERVAL = 180

# 模型保存到本地磁盘的间隔 (秒)
SAVE_EVERY_SEC = 180
# =====================================================

# 1. 检测当前已有的步数
current_step = 0
metrics_file = output_dir / "metrics.jsonl"
if metrics_file.exists():
    try:
        with open(metrics_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    record = json.loads(line)
                    current_step = int(record.get("step", current_step))
    except Exception as e:
        print(f"[提示] 读取历史 metrics 时异常: {e}")

print("=" * 60)
print(f"📊 当前存档进度:     {current_step:,} steps")
print(f"🎯 设定目标总步数:   {TARGET_STEPS:,} steps")

if TARGET_STEPS <= current_step:
    print("\n" + "!" * 60)
    print(f"【提醒】设定目标步数 ({TARGET_STEPS:,}) <= 当前已完成步数 ({current_step:,})！")
    print(f"DreamerV3 将认为训练已达标而直接退出。")
    auto_new_target = current_step + 50000
    print(f"👉 自动为您将目标步数递增至: {auto_new_target:,} 步 (增训 50,000 步)")
    print("!" * 60 + "\n")
    TARGET_STEPS = auto_new_target

incremental_steps = TARGET_STEPS - current_step
print(f"🚀 本次将执行增量训练: {incremental_steps:,} steps")
if in_colab and gdrive_dir.parent.exists():
    print(f"🛡️ 途中自动云盘备份: 已启用 (后台每 {AUTO_BACKUP_INTERVAL} 秒无阻增量同步至 Google Drive)")
else:
    print(f"💾 本地自动检查点存储: 已启用 (每 {SAVE_EVERY_SEC} 秒写入磁盘)")
print("=" * 60 + "\n")

# 2. 构造 DreamerV3 运行命令
import jax
has_gpu_device = any(d.platform == 'gpu' for d in jax.devices())
platform_arg = 'cuda' if has_gpu_device else 'cpu'

main_script = repo_dir / "dreamerv3" / "main.py"
cmd = [
    sys.executable,
    str(main_script),
    "--configs", "crafter", MODEL_SIZE,
    "--task", "crafter_reward",
    "--logdir", str(output_dir),
    "--run.steps", str(TARGET_STEPS),
    "--run.train_ratio", str(TRAIN_RATIO),
    "--run.log_every", "30",
    "--run.save_every", str(SAVE_EVERY_SEC),
    "--jax.platform", platform_arg,
    "--jax.prealloc", "False",
]

env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir) + os.pathsep + env.get("PYTHONPATH", "")
# 确保清理已废弃的 XLA 参数，防止新版 XLA 崩溃
env.pop("XLA_FLAGS", None)

print(f"[执行命令] {' '.join(cmd)}\n")

# 3. 启动【途中后台自动备份守护线程】(若云盘可用)
stop_backup_flag = threading.Event()

def periodic_backup_worker():
    while not stop_backup_flag.is_set():
        if stop_backup_flag.wait(timeout=AUTO_BACKUP_INTERVAL):
            break
        if gdrive_dir.parent.exists() and output_dir.exists():
            try:
                shutil.copytree(output_dir, gdrive_dir, dirs_exist_ok=True)
                now_str = time.strftime("%H:%M:%S")
                step_tag = ""
                mf = output_dir / "metrics.jsonl"
                if mf.exists():
                    try:
                        with open(mf, "r", encoding="utf-8") as f:
                            lines = [l.strip() for l in f if l.strip()]
                            if lines:
                                cur_s = json.loads(lines[-1]).get("step", "")
                                if cur_s:
                                    step_tag = f" | 步数 ~{int(cur_s):,} steps"
                    except Exception:
                        pass
                print(f"\n[途中自动备份 {now_str}] 💾 最新权重与指标已成功增量同步至 Google 云盘{step_tag}！", flush=True)
            except Exception as e:
                pass

backup_thread = threading.Thread(target=periodic_backup_worker, daemon=True)
backup_thread.start()

# 4. 启动并流式监听训练
try:
    proc = subprocess.Popen(
        cmd,
        cwd=str(repo_dir),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    return_code = proc.wait()
    if return_code != 0:
        print(f"\n[警告] 训练进程退出，返回码: {return_code}")
    else:
        print(f"\n🎉 恭喜！训练成功达成目标步数 ({TARGET_STEPS:,} 步)！")
except KeyboardInterrupt:
    print("\n[中断] 接收到手动中断信号 (Ctrl+C)。正在终止训练并保护检查点...")
    if 'proc' in locals():
        proc.terminate()
        proc.wait()
finally:
    stop_backup_flag.set()
    backup_thread.join(timeout=5)
    if in_colab and gdrive_dir.parent.exists() and output_dir.exists():
        print("\n[最终备份] 正在将最终检查点、经验池与评估日志同步至 Google Drive...")
        try:
            shutil.copytree(output_dir, gdrive_dir, dirs_exist_ok=True)
            print(f"✅ Google Drive 同步成功: {gdrive_dir}")
        except Exception as e:
            print(f"[备份警告] 最终同步至云盘失败: {e}")


📊 当前存档进度:     64,920 steps
🎯 设定目标总步数:   100,000 steps
🚀 本次将执行增量训练: 35,080 steps
🛡️ 途中自动云盘备份: 已启用 (后台每 180 秒无阻增量同步至 Google Drive)

[执行命令] /usr/bin/python3 /content/dreamerv3/dreamerv3/main.py --configs crafter size12m --task crafter_reward --logdir /content/output/crafter_dreamerv3_100k --run.steps 100000 --run.train_ratio 512 --run.log_every 30 --run.save_every 180 --jax.platform cuda --jax.prealloc False

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/content/dreamerv3/embodied/jax/agent.py:112: DeprecationWarning: `with mesh:` context manager has been deprecated. Please use `with jax.set_mesh(mesh):` instead.
  with self.train_mesh:
/usr/local/lib/python3.13/dist-packages/

## 6. 模型检查点与对局经验全景看板
统计当前工作区目录（或 Google Drive）中的实际资产文件、模型权重体积、对局经验池数量与最新评估指标。


In [ ]:
import os
import json
import time
import shutil
from pathlib import Path

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    src_dir = Path("/content/output/crafter_dreamerv3_100k")
    dst_dir = Path("/content/drive/MyDrive/crafter_dreamerv3_100k")
else:
    cur = Path.cwd()
    project_root = cur if (cur / "colab").exists() else cur.parent
    src_dir = project_root / "output" / "crafter_dreamerv3_100k"
    dst_dir = src_dir

print("=" * 65)
print(" 🛡️ 训练资产与模型检查点全景看板")
print("=" * 65)

scan_dir = dst_dir if dst_dir.exists() else src_dir
if scan_dir.exists():
    all_files = list(scan_dir.rglob("*"))
    files_only = [f for f in all_files if f.is_file()]
    
    total_size_mb = sum(f.stat().st_size for f in files_only) / (1024 * 1024)
    print(f"📁 存储目录: {scan_dir}")
    print(f"📦 资产统计: 共计 {len(files_only):,} 个文件，占用空间: {total_size_mb:.2f} MB\n")
    
    # 识别模型检查点
    model_files = [f for f in files_only if any(k in f.name.lower() for k in ["checkpoint", "agent", ".ckpt", ".pkl"])]
    if model_files:
        print(f"🤖 【模型检查点 (Checkpoints / Weights)】: 共 {len(model_files)} 个核心文件")
        for mf in sorted(model_files, key=lambda x: -x.stat().st_mtime)[:10]:
            sz_mb = mf.stat().st_size / (1024 * 1024)
            mt = time.strftime("%H:%M:%S", time.localtime(mf.stat().st_mtime))
            rel_p = mf.relative_to(scan_dir)
            print(f"  • {str(rel_p):<35} ({sz_mb:6.2f} MB, 更新于 {mt})")
    else:
        print("🤖 【模型检查点】: 当前正在训练中或等待写入独立权重文件...")

    # 识别经验回放数据
    replay_files = [f for f in files_only if "replay" in str(f).lower() or f.suffix in [".npz", ".chunk"]]
    if replay_files:
        print(f"\n🎮 【对局经验池 (Replay Buffer)】: 发现 {len(replay_files):,} 个经验块/轨迹文件")
        for rf in sorted(replay_files, key=lambda x: -x.stat().st_mtime)[:5]:
            sz_mb = rf.stat().st_size / (1024 * 1024)
            mt = time.strftime("%H:%M:%S", time.localtime(rf.stat().st_mtime))
            rel_p = mf.relative_to(scan_dir)
            print(f"  • {str(rel_p):<35} ({sz_mb:6.2f} MB, 更新于 {mt})")
        if len(replay_files) > 5:
            print(f"  ... 以及其他 {len(replay_files) - 5} 个历史对局轨迹文件")

    # 提取评测指标
    metrics_file = scan_dir / "metrics.jsonl"
    if metrics_file.exists():
        records_count = 0
        latest_step = 0
        latest_score = "N/A"
        latest_length = "N/A"
        
        with open(metrics_file, "r", encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]
            records_count = len(lines)
            for l in reversed(lines):
                try:
                    d = json.loads(l)
                    if latest_step == 0 and "step" in d:
                        latest_step = int(d["step"])
                    s = d.get("episode/score", d.get("score"))
                    l_val = d.get("episode/length", d.get("length"))
                    if s is not None and latest_score == "N/A":
                        val_f = float(s)
                        if not (val_f != val_f):
                            latest_score = f"{val_f:.2f}"
                    if l_val is not None and latest_length == "N/A":
                        len_f = float(l_val)
                        if not (len_f != len_f):
                            latest_length = f"{len_f:.1f}"
                    if latest_step > 0 and latest_score != "N/A" and latest_length != "N/A":
                        break
                except Exception:
                    pass

        print(f"\n📈 【训练指标概览】: 共计 {records_count:,} 条评估记录")
        print(f"  • 最新训练步数:   {latest_step:,} steps")
        print(f"  • 最新平均得分:   {latest_score}")
        print(f"  • 最新存活时长:   {latest_length} steps")
    
    print("\n" + "=" * 65)
    print("🎉 资产核验完毕！数据已 100% 保持最新状态。")
    print("=" * 65)
else:
    print(f"未在目标目录找到数据文件: {scan_dir}")


## 7. 训练成果评估与 22 项成就解锁可视化
动态解析 `metrics.jsonl`，绘制最新累计步数的平均得分曲线、存活时长曲线以及 Crafter 22 项成就解锁概率柱状图。


In [3]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    search_dir = Path("/content/output/crafter_dreamerv3_100k")
else:
    cur = Path.cwd()
    project_root = cur if (cur / "colab").exists() else cur.parent
    search_dir = project_root / "output" / "crafter_dreamerv3_100k"

metrics_file = search_dir / "metrics.jsonl"
if not metrics_file.exists():
    files = list(search_dir.glob("**/metrics.jsonl"))
    if files:
        metrics_file = files[0]

if metrics_file.exists():
    records = []
    with open(metrics_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass

    df = pd.DataFrame(records)
    print(f"成功载入 {len(df)} 条指标记录！")

    last_row = df.iloc[-1]
    latest_step = int(last_row.get("step", 0))
    latest_score = float(last_row.get("episode/score", last_row.get("score", 0.0)))
    latest_len = float(last_row.get("episode/length", last_row.get("length", 0.0)))

    print(f"📊 最新总训练步数: {latest_step:,} steps")
    print(f"🏆 当前最新平均分: {latest_score:.2f}")
    print(f"⏱️ 当前平均存活长: {latest_len:.1f} steps")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    score_col = "episode/score" if "episode/score" in df.columns else "score"
    len_col = "episode/length" if "episode/length" in df.columns else "length"

    if score_col in df.columns and "step" in df.columns:
        axes[0].plot(df["step"], df[score_col], color="#2563EB", linewidth=2, label="Score")
        axes[0].set_title(f"Crafter Score vs Steps (Up to {latest_step:,})", fontsize=13, fontweight="bold")
        axes[0].set_xlabel("Steps")
        axes[0].set_ylabel("Score")
        axes[0].grid(True, linestyle="--", alpha=0.5)
        axes[0].legend()

    if len_col in df.columns and "step" in df.columns:
        axes[1].plot(df["step"], df[len_col], color="#10B981", linewidth=2, label="Episode Length")
        axes[1].set_title(f"Episode Survival Length (Up to {latest_step:,})", fontsize=13, fontweight="bold")
        axes[1].set_xlabel("Steps")
        axes[1].set_ylabel("Survival Length (Steps)")
        axes[1].grid(True, linestyle="--", alpha=0.5)
        axes[1].legend()

    plt.tight_layout()
    plt.show()

    # 提取 Crafter 成就达成情况
    achievements = {}
    for col in df.columns:
        if "achievement" in col.lower():
            ach_name = col.split("/")[-1].replace("achievement_", "")
            achievements[ach_name] = float(last_row[col])

    if achievements:
        ach_series = pd.Series(achievements).sort_values(ascending=True)
        plt.figure(figsize=(10, 8))
        colors = ["#10B981" if v > 0.05 else "#9CA3AF" for v in ach_series.values]
        bars = plt.barh(ach_series.index, ach_series.values * 100, color=colors)
        plt.title(f"Crafter Achievement Unlock Rate at {latest_step:,} Steps (%)", fontsize=14, fontweight="bold")
        plt.xlabel("Unlock Probability (%)")
        plt.grid(axis="x", linestyle="--", alpha=0.6)
        for bar in bars:
            w = bar.get_width()
            if w > 1:
                plt.text(w + 1, bar.get_y() + bar.get_height() / 2, f"{w:.1f}%", va="center", fontsize=9)
        plt.tight_layout()
        plt.show()

        unlocked = sum(1 for v in achievements.values() if v > 0.05)
        print(f"\n🎉 累计解锁成就项: {unlocked} / {len(achievements)}")
else:
    print(f"未在 {search_dir} 找到 metrics.jsonl 文件，请先执行训练或同步历史存档。")


未在 /content/output/crafter_dreamerv3_100k 找到 metrics.jsonl 文件，请先执行训练或同步历史存档。
